In [ ]:
import numpy as np
import csv
import hashlib
from pathlib import Path

from PIL import Image
from google.colab import files
import matplotlib.pyplot as plt


# Configuration

KEY = 0xC3
NUMBER = 200

VECTOR = [
    10, 200, 55, 128,
    0, 255, 77, 33,
    9, 250, 61, 190,
    5, 222, 100, 47
]

IMAGE_WIDTH = 256
IMAGE_HEIGHT = 256

OUTPUT_DIR = Path("xor_results")
OUTPUT_DIR.mkdir(exist_ok=True)


# Basic XOR function

def xor_byte(value, key=KEY):
    """XOR an 8-bit value with the key."""

    if not 0 <= value <= 255:
        raise ValueError("Value must be between 0 and 255.")

    return value ^ key


# Part 1: Single 8-bit number

def run_part1():
    print("\nPART 1: SINGLE 8-BIT NUMBER")

    plaintext = NUMBER

    ciphertext = xor_byte(plaintext)
    decrypted = xor_byte(ciphertext)

    print(f"Key       : {KEY} / 0x{KEY:02X} / {KEY:08b}")
    print(f"Plaintext : {plaintext} / 0x{plaintext:02X} / {plaintext:08b}")
    print(f"Ciphertext: {ciphertext} / 0x{ciphertext:02X} / {ciphertext:08b}")
    print(f"Decrypted : {decrypted} / 0x{decrypted:02X} / {decrypted:08b}")

    assert ciphertext == 11
    assert decrypted == plaintext

    print("Result: PASS")

    return plaintext, ciphertext, decrypted


# Part 2: 16-byte vector

def run_part2():
    print("\nPART 2: 16-BYTE VECTOR")

    plaintext = VECTOR.copy()

    ciphertext = [xor_byte(value) for value in plaintext]
    decrypted = [xor_byte(value) for value in ciphertext]

    print("Plaintext:")
    print(plaintext)

    print("\nCiphertext:")
    print(ciphertext)

    print("\nDecrypted:")
    print(decrypted)

    expected_ciphertext = [
        201, 11, 244, 67,
        195, 60, 142, 226,
        202, 57, 254, 125,
        198, 29, 167, 236
    ]

    assert ciphertext == expected_ciphertext
    assert decrypted == plaintext

    print("Result: PASS")

    return plaintext, ciphertext, decrypted


# Part 3: Image input and validation

def upload_test_image():
    print("\nPART 3: 256x256 GRAYSCALE IMAGE")
    print("Upload the 256x256 grayscale test image.")

    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("No image was uploaded.")

    filename = list(uploaded.keys())[0]

    print(f"Uploaded: {filename}")

    return filename


def load_and_validate_image(filename):
    image = Image.open(filename)

    print("\nIMAGE VERIFICATION")
    print(f"Original format : {image.format}")
    print(f"Original mode   : {image.mode}")
    print(f"Original size   : {image.size}")

    image = image.convert("L")
    pixels = np.array(image, dtype=np.uint8)

    height, width = pixels.shape

    print(f"Width           : {width}")
    print(f"Height          : {height}")
    print("Channels        : 1")
    print("Type            : 8-bit Grayscale")
    print(f"Pixel range     : {pixels.min()} - {pixels.max()}")
    print(f"Total pixels    : {pixels.size}")

    assert width == IMAGE_WIDTH
    assert height == IMAGE_HEIGHT
    assert pixels.shape == (256, 256)
    assert pixels.dtype == np.uint8
    assert pixels.size == 65536

    print("Image validation: PASS")

    return pixels


# Image encryption

def encrypt_image(image_array):
    return np.bitwise_xor(
        image_array,
        KEY
    ).astype(np.uint8)


# Image decryption

def decrypt_image(encrypted_array):
    return np.bitwise_xor(
        encrypted_array,
        KEY
    ).astype(np.uint8)


# Save image

def save_image(array, filename):
    Image.fromarray(array).save(
        OUTPUT_DIR / filename
    )


# SHA-256 hash

def calculate_hash(array):
    return hashlib.sha256(
        array.tobytes()
    ).hexdigest()


# Generate image test vectors

def generate_test_vectors(original, encrypted, decrypted):
    filename = OUTPUT_DIR / "image_test_vectors.csv"

    original_flat = original.flatten()
    encrypted_flat = encrypted.flatten()
    decrypted_flat = decrypted.flatten()

    with open(filename, "w", newline="") as file:
        writer = csv.writer(file)

        writer.writerow([
            "index",
            "row",
            "column",
            "plaintext",
            "ciphertext",
            "decrypted"
        ])

        for index in range(65536):
            row = index // 256
            column = index % 256

            writer.writerow([
                index,
                row,
                column,
                int(original_flat[index]),
                int(encrypted_flat[index]),
                int(decrypted_flat[index])
            ])

    print(f"\nTest-vector file created: {filename}")

    return filename


# Run image encryption and decryption

def run_part3(filename):
    original = load_and_validate_image(filename)

    print("\nEncrypting image...")
    encrypted = encrypt_image(original)

    print("Decrypting image...")
    decrypted = decrypt_image(encrypted)

    save_image(original, "original_python.png")
    save_image(encrypted, "encrypted_python.png")
    save_image(decrypted, "decrypted_python.png")

    # Pixel-by-pixel verification

    mismatch_mask = original != decrypted

    mismatch_count = int(
        np.count_nonzero(mismatch_mask)
    )

    max_difference = int(
        np.max(
            np.abs(
                original.astype(np.int16)
                - decrypted.astype(np.int16)
            )
        )
    )

    # Verify encryption

    expected_ciphertext = np.bitwise_xor(
        original,
        KEY
    ).astype(np.uint8)

    ciphertext_match = np.array_equal(
        encrypted,
        expected_ciphertext
    )

    # SHA-256 verification

    original_hash = calculate_hash(original)
    decrypted_hash = calculate_hash(decrypted)

    hash_match = original_hash == decrypted_hash

    print("\nIMAGE RESULTS")
    print(f"Total pixels       : {original.size}")
    print(f"Different pixels   : {mismatch_count}")
    print(f"Maximum difference : {max_difference}")
    print(f"Ciphertext check   : {'PASS' if ciphertext_match else 'FAIL'}")
    print(f"Decryption check   : {'PASS' if mismatch_count == 0 else 'FAIL'}")
    print(f"Hash check         : {'PASS' if hash_match else 'FAIL'}")

    assert ciphertext_match
    assert mismatch_count == 0
    assert max_difference == 0
    assert hash_match

    vector_file = generate_test_vectors(
        original,
        encrypted,
        decrypted
    )

    # Display original, encrypted and decrypted images

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(
        original,
        cmap="gray",
        vmin=0,
        vmax=255
    )
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(
        encrypted,
        cmap="gray",
        vmin=0,
        vmax=255
    )
    plt.title("Encrypted")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(
        decrypted,
        cmap="gray",
        vmin=0,
        vmax=255
    )
    plt.title("Decrypted")
    plt.axis("off")

    plt.tight_layout()

    comparison_file = OUTPUT_DIR / "image_comparison.png"

    plt.savefig(
        comparison_file,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    print("\nGenerated files:")
    print(OUTPUT_DIR / "original_python.png")
    print(OUTPUT_DIR / "encrypted_python.png")
    print(OUTPUT_DIR / "decrypted_python.png")
    print(vector_file)
    print(comparison_file)

    print("\nResult: PASS")

    return original, encrypted, decrypted


# Main program

print("\n8-BIT XOR CIPHER - PYTHON REFERENCE")

part1_results = run_part1()
part2_results = run_part2()

uploaded_filename = upload_test_image()
part3_results = run_part3(uploaded_filename)

print("\nFINAL RESULT")
print("Part 1 - Number : PASS")
print("Part 2 - Vector : PASS")
print("Part 3 - Image  : PASS")
print("\nAll Python implementations completed successfully.")


8-BIT XOR CIPHER - PYTHON REFERENCE

PART 1: SINGLE 8-BIT NUMBER
Key       : 195 / 0xC3 / 11000011
Plaintext : 200 / 0xC8 / 11001000
Ciphertext: 11 / 0x0B / 00001011
Decrypted : 200 / 0xC8 / 11001000
Result: PASS

PART 2: 16-BYTE VECTOR
Plaintext:
[10, 200, 55, 128, 0, 255, 77, 33, 9, 250, 61, 190, 5, 222, 100, 47]

Ciphertext:
[201, 11, 244, 67, 195, 60, 142, 226, 202, 57, 254, 125, 198, 29, 167, 236]

Decrypted:
[10, 200, 55, 128, 0, 255, 77, 33, 9, 250, 61, 190, 5, 222, 100, 47]
Result: PASS

PART 3: 256x256 GRAYSCALE IMAGE
Upload the 256x256 grayscale test image.
